# ⚡ Colab Ollama MCP — Option 1: Saver Multi-Vendor (Tesla T4 16GB · Siêu Tiết Kiệm)

> **Chế độ dự phòng tiết kiệm:** Sử dụng khi CU gần chạm giới hạn hoặc khi cần tốc độ phản hồi cực nhanh (60+ tok/s). Kết hợp tinh hoa đa hãng $\le 8B$ (DeepSeek + Meta + Alibaba), máy cực mát, VRAM luôn dư dả >50%.

---

## 🧩 Bộ Ba Model Đa Hãng Theo Scope (Option 1):

| Scope | Model Tag | Hãng | VRAM | Điểm mạnh cốt lõi |
| :--- | :--- | :--- | :---: | :--- |
| **1. Planning** | **`deepseek-r1:8b`** | **DeepSeek** | ~5.2 GB | Suy luận chuỗi tư duy `<think>` bóc tách yêu cầu và lên checklist task. |
| **2. Workflow** | **`llama3.1:8b`** | **Meta** | ~4.9 GB | Tối ưu hóa đặc biệt cho Tool Use / Function Calling & điều phối pipeline. |
| **3. Coding & Diagram** | **`qwen2.5-coder:7b`** | **Alibaba** | ~4.5 GB | Tốc độ cực nhanh 65 tok/s, code boilerplate & vẽ sơ đồ Mermaid.js chuẩn xác. |

---

💡 **Khi cần chất lượng code cao cấp hơn (Option 2):** Mở notebook [`colab_ollama_mcp.ipynb`](colab_ollama_mcp.ipynb)!

### Step 1: Xác Thực Phần Cứng NVIDIA GPU (Tesla T4 16GB)
Kiểm tra card GPU Tesla T4 được gắn vào máy ảo Colab.

In [ ]:
!nvidia-smi

### Step 2: Cấu Hình Lưu Trữ Mô Hình (Colab NVMe SSD Siêu Tốc)
Sử dụng ổ cứng NVMe nội bộ của Colab để nạp và hoán đổi model với tốc độ cao nhất (>600 MB/s).

In [ ]:
import os
USE_GDRIVE = False

if USE_GDRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        models_dir = '/content/drive/MyDrive/ollama_models'
        os.environ['OLLAMA_MODELS'] = models_dir
        os.makedirs(models_dir, exist_ok=True)
        print(f"✅ Persistent Google Drive: {models_dir}")
    except Exception as e:
        print(f"⚡ Sử dụng Colab SSD nội bộ (/root/.ollama/models).")
else:
    print("⚡ Sử dụng Colab NVMe SSD nội bộ (/root/.ollama/models) — Nạp model siêu tốc.")

### Step 3: Cài Đặt Ollama & Khởi Động Daemon
Tải bản Linux Ollama và khởi chạy nền trên cổng 11434 với cấu hình tối ưu chống tràn VRAM.

In [ ]:
import os
import subprocess
import time
import urllib.request
from IPython.display import clear_output

# 1. Kích hoạt 16GB Linux NVMe Swapfile (Nâng System RAM từ 12.67GB lên 28.67GB, chống OOM 100%)
!test -f /swapfile || (fallocate -l 16G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile)

# 2. Cài đặt zstd và bản phát hành Ollama Linux chính thức
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Cấu hình biến môi trường tối ưu VRAM & chống tràn bộ nhớ
env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"
env["OLLAMA_KEEP_ALIVE"] = "24h"          # Giữ model trong VRAM liên tục
env["OLLAMA_MAX_LOADED_MODELS"] = "1"      # Đảm bảo VRAM chỉ gánh 1 model lớn tại 1 thời điểm
env["OLLAMA_FLASH_ATTENTION"] = "1"       # Bật Flash Attention tăng tốc sinh token
env["OLLAMA_KV_CACHE_TYPE"] = "q8_0"      # Nén KV Cache 8-bit, giảm 50% dung lượng VRAM

# 4. Khởi chạy 'ollama serve' ngầm
ollama_proc = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
clear_output(wait=True)
time.sleep(3)

try:
    with urllib.request.urlopen("http://localhost:11434/") as resp:
        print("🎉 Ollama Daemon đã sẵn sàng (NVMe Swap 16GB + Flash Attention + KV Cache q8_0):")
        print(resp.read().decode("utf-8"))
except Exception as e:
    print("⚠️ Không thể kết nối Ollama daemon:", e)


### Step 4: Tải Bộ Ba Model Đa Hãng Tiết Kiệm (Option 1)
Tải trước 3 model nhỏ gọn chuyên biệt cho từng scope:
1. `deepseek-r1:8b` (Planning & Deep CoT Reasoning)
2. `llama3.1:8b` (Workflow & Tool Calling)
3. `qwen2.5-coder:7b` (Fast Coding & Mermaid Diagram)

In [ ]:
from IPython.display import clear_output

# Bộ ba mô hình Option 1 (Saver Multi-Vendor)
SAVER_MODELS = [
    ("deepseek-r1:8b",   "🧠 Planning: DeepSeek CoT Reasoning (<think> mode)"),
    ("llama3.1:8b",       "🔄 Workflow: Meta Tool Calling & Orchestration"),
    ("qwen2.5-coder:7b", "💻 Coding & Diagram: Alibaba High-Speed (65 tok/s)"),
]

print("⏳ Đang tải bộ ba model Option 1 (Saver Multi-Vendor)...")
for tag, role in SAVER_MODELS:
    print(f"\n📦 Đang tải {tag} ({role})...")
    !ollama pull {tag}

clear_output(wait=True)
print("✅ Hoàn tất tải toàn bộ mô hình! Danh sách hiện có trong máy:")
!ollama list


### Step 5: Mở Cloudflare Quick Tunnel & Kết Nối Với Antigravity IDE
Khởi tạo reverse tunnel HTTPS an toàn kết nối về máy ảo Colab.

In [ ]:
import re
import subprocess
import time
from IPython.display import clear_output

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:11434"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

tunnel_url = None
print("⏳ Đang thiết lập Cloudflare Quick Tunnel...")
for _ in range(60):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(0.2)

clear_output(wait=True)
if tunnel_url:
    print("=" * 78)
    print("🎉 COLAB OLLAMA SAVER TUNNEL IS ONLINE!")
    print(f"🔗 PUBLIC ENDPOINT URL: {tunnel_url}")
    print("=" * 78)
    print("\n🚀 CÁCH 1 (Dán trực tiếp URL vào đoạn chat Antigravity IDE):\n")
    print(f"   {tunnel_url}\n")
    print("⚡ CÁCH 2 (Chạy lệnh trong Terminal máy Mac):\n")
    print(f"   python3 /Users/marchhygge/MarchHygge/project/colab-ollama-mcp/colab_ollama_mcp/server.py --set-endpoint {tunnel_url}")
    print("\n" + "-" * 78)
else:
    print("❌ Không thể tạo đường hầm Cloudflare. Vui lòng chạy lại cell này.")

### Step 6: Bảng Theo Dõi Tài Nguyên Thời Gian Thực (GPU VRAM, System RAM & Model Đang Nạp)
Chạy cell này để kiểm tra tài nguyên và model đang được ghim trong GPU VRAM.

In [ ]:
import json
import shutil
import subprocess
import time
import urllib.request
from IPython.display import clear_output

AUTO_REFRESH = False
REFRESH_SECONDS = 5

def make_bar(pct: float, width: int = 24) -> str:
    filled = max(0, min(width, int(round((pct / 100.0) * width))))
    return "█" * filled + "░" * (width - filled)

def print_saver_telemetry() -> None:
    print("=" * 76)
    print(f"📊 COLAB SAVER SERVER TELEMETRY — {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}")
    print("=" * 76)
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=name,temperature.gpu,utilization.gpu,memory.used,memory.total", "--format=csv,noheader,nounits"], text=True).strip()
        for line in out.splitlines():
            name, temp, util, mem_used, mem_total = [x.strip() for x in line.split(",")]
            used_gb = float(mem_used) / 1024.0
            total_gb = float(mem_total) / 1024.0
            vram_pct = (float(mem_used) / float(mem_total)) * 100.0 if float(mem_total) > 0 else 0.0
            print(f"🎮 GPU Hardware : {name}  (🌡️ {temp}°C)")
            print(f"🧠 GPU VRAM     : [{make_bar(vram_pct)}] {vram_pct:5.1f}%  ({used_gb:.2f} GB / {total_gb:.2f} GB)")
    except Exception as e:
        print(f"⚠️ GPU Status   : {e}")
    try:
        with urllib.request.urlopen("http://localhost:11434/api/ps", timeout=5) as resp:
            ps_data = json.loads(resp.read().decode("utf-8"))
            running = ps_data.get("models", [])
            print("-" * 76)
            if running:
                for m in running:
                    vram_gb = m.get("size_vram", 0) / (1024 ** 3)
                    print(f"🔥 Đang chạy trong VRAM: {m.get('name')} ({vram_gb:.2f} GB)")
            else:
                print("💤 VRAM trống (Model nạp tức thì trong ~0.8s khi gửi prompt)")
    except Exception as e:
        print(f"🔴 Ollama Status: {e}")
    print("=" * 76)

try:
    while True:
        clear_output(wait=True)
        print_saver_telemetry()
        if not AUTO_REFRESH:
            break
        time.sleep(REFRESH_SECONDS)
except KeyboardInterrupt:
    print("⏹️ Dừng theo dõi.")